# Monocle3 — *C. elegans* L2 (Python port)

Python port of the R vignette `c_elegans_L2_v2.ipynb`. Covers preprocessing, `plot_pc_variance_explained`, batch correction, clustering, marker detection, partition → cell-type assignment, and Garnett marker file generation on the Cao et al. 2017 L2 worm atlas (42035 cells × 20271 genes).


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd

import monocle3 as m3
print('monocle3-python', m3.__version__)

## 2. Load L2 worm data (Cao et al. 2017)

In [ ]:
adata = m3.load_cao_l2()
m3.estimate_size_factors(adata)
adata

## 3. Preprocess (size-factor + PCA) and inspect variance

In [ ]:
m3.preprocess_cds(adata, num_dim=100)
m3.plot_pc_variance_explained(adata)

## 4. UMAP (before batch correction) — shows the `plate` batch effect

In [ ]:
m3.reduce_dimension(adata)
m3.plot_cells(adata, color_cells_by='cao_cell_type')

In [ ]:
m3.plot_cells(adata, color_cells_by='plate', label_cell_groups=False)

In [ ]:
m3.plot_cells(adata, genes=['cpna-2', 'egl-21', 'ram-2', 'inos-1'])

## 5. Align on `plate` and re-embed — batch effect is removed

In [ ]:
m3.align_cds(adata, alignment_group='plate')
m3.reduce_dimension(adata, umap_fast_sgd=False, cores=1)
m3.plot_cells(adata, color_cells_by='plate', label_cell_groups=False)

In [ ]:
m3.plot_cells(adata, color_cells_by='cao_cell_type', label_groups_by_cluster=False)

## 6. Cluster cells + partition-level structure

In [ ]:
m3.cluster_cells(adata)
m3.plot_cells(adata, color_cells_by='partition', group_cells_by='partition')

In [ ]:
m3.plot_cells(adata, color_cells_by='cao_cell_type')

## 7. Top marker genes per partition

In [ ]:
marker_test_res = m3.top_markers(
    adata, group_cells_by='partition', reference_cells=1000, cores=8,
)
top_specific_markers = (
    marker_test_res.query('fraction_expressing >= 0.10')
    .sort_values('pseudo_R2', ascending=False)
    .groupby('cell_group', as_index=False)
    .head(3)
)
top_specific_marker_ids = pd.unique(top_specific_markers['gene_id']).tolist()

m3.plot_genes_by_group(
    adata, top_specific_marker_ids,
    group_cells_by='partition',
    ordering_type='cluster_row_col',
    max_size=3,
)

## 8. Auto-label partitions by majority `cao_cell_type`

The upstream tutorial hand-recodes partition IDs 1–34 into cell-type names. Our clustering won't reproduce those IDs, so we pick the majority `cao_cell_type` per partition programmatically.

In [ ]:
parts = m3.partitions(adata).astype(str)
cross = pd.crosstab(parts, adata.obs['cao_cell_type'].astype(str))
cross_prop = cross.div(cross.sum(axis=1), axis=0).fillna(0)
partition_to_type = cross_prop.idxmax(axis=1).to_dict()

adata.obs['assigned_cell_type'] = parts.map(partition_to_type)
m3.plot_cells(adata, group_cells_by='partition', color_cells_by='assigned_cell_type')

## 9. Re-run `top_markers` on `assigned_cell_type` → Garnett marker file

In [ ]:
assigned_mask = adata.obs['assigned_cell_type'].notna().to_numpy()
assigned_adata = adata[assigned_mask].copy()
assigned_type_marker_test_res = m3.top_markers(
    assigned_adata,
    group_cells_by='assigned_cell_type',
    reference_cells=1000, cores=8,
)

garnett_markers = (
    assigned_type_marker_test_res.query(
        'marker_test_q_value < 0.01 & specificity >= 0.5'
    )
    .sort_values('marker_score', ascending=False)
    .groupby('cell_group', as_index=False)
    .head(5)
)
# Remove duplicates that mark multiple groups.
garnett_markers = garnett_markers.groupby('gene_short_name', as_index=False) \
    .filter(lambda g: len(g) == 1)

m3.generate_garnett_marker_file(garnett_markers, file='./marker_file.txt')